# 第2章　自動微分（Autograd）— 学習の心臓

ニューラルネットの「学習」とは、**誤差（損失）を小さくする方向にパラメータを少しずつ動かす**こと。
その「方向」を教えてくれるのが**勾配（こうばい / gradient）**で、PyTorch はこれを自動計算します。これが Autograd。

この章のゴール：`backward()` が何をしているか説明でき、**勾配降下法を自分の手で**実装できる。

> **このノートの使い方**
> - 上から順にセルを実行（Colab/Jupyter ともに `Shift + Enter`）。
> - コードは**少し書き換えて壊して直す**のが一番伸びます。各章末に演習があります。
> - GPU は不要な章が多いです。重い章（CNN）では使い方を案内します。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 2-0. 数学ミニ復習（忘れてOK、ここで思い出す）

- **微分＝傾き**：関数 $f(x)$ の微分 $f'(x)$ は「$x$ を少し動かしたとき $f$ がどれだけ変化するか」。傾きが正なら右肩上がり、負なら右肩下がり。
- **最小値の方へ動かす**：$f$ を小さくしたいなら、傾きと**逆向き**に $x$ を動かせばよい。
  $$x \leftarrow x - \eta\, f'(x)$$
  ここで $\eta$（イータ）は**学習率**＝1歩の大きさ。これが**勾配降下法**。
- **勾配（gradient）**：変数が複数あるとき（$w_1, w_2, \dots$）の「各方向の傾き」をまとめたもの $\nabla f$。
- **連鎖律（chain rule）**：合成関数の微分。$y=f(g(x))$ なら $\dfrac{dy}{dx}=\dfrac{dy}{dg}\cdot\dfrac{dg}{dx}$。
  ニューラルネットは関数の入れ子なので、連鎖律を端から掛け算していくだけ＝**誤差逆伝播（backprop）**。PyTorch が自動でやります。

## 2-1. `requires_grad` と `backward()`

`requires_grad=True` を付けたテンソルは「追跡対象」になり、それを使った計算が記録されます。
最後の値で `.backward()` を呼ぶと、各変数の `.grad` に勾配が入ります。

In [ ]:
import torch

w = torch.tensor(3.0, requires_grad=True)   # 勾配を追跡
y = w ** 2                                  # y = w^2
y.backward()                                # dy/dw を計算
print("w.grad =", w.grad)                   # 2*w = 6.0（手計算と一致）

手計算：$y=w^2$ なら $\dfrac{dy}{dw}=2w$。$w=3$ で $6$。PyTorch の答えと一致しますね。

### 連鎖律の例
$y = (2w+1)^2$ を $w=1$ で。手計算：$\frac{dy}{dw}=2(2w+1)\cdot 2 = 4(2w+1) = 12$。

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
u = 2 * w + 1
y = u ** 2
y.backward()
print("w.grad =", w.grad)   # 12.0

## 2-2. 勾配降下を「手で」実装する（最重要・直感の核）

$f(x) = (x-3)^2$ を最小にする $x$ を探します。答えは明らかに $x=3$ ですが、
**勾配だけを頼りに少しずつ近づける**過程を体験します。これが全学習の縮図です。

In [ ]:
x = torch.tensor(0.0, requires_grad=True)   # 適当な初期値から出発
lr = 0.1                                      # 学習率（1歩の大きさ）

for step in range(20):
    f = (x - 3) ** 2          # 最小化したい関数（=損失のつもり）
    f.backward()              # df/dx を計算 -> x.grad に入る

    with torch.no_grad():     # 更新は追跡しない（下で説明）
        x -= lr * x.grad      # 勾配と逆向きに1歩進む
    x.grad.zero_()            # 次に備えて勾配をリセット（重要！）

    if step % 4 == 0:
        print(f"step {step:2d}: x={x.item():.4f}, f={f.item():.4f}")

print("最終 x =", x.item(), "（3 に近づくはず）")

上のループの中身が、後の章の「学習ループ」とそっくりです：
**① 損失を計算 → ② `backward()` で勾配 → ③ 勾配と逆向きに更新 → ④ 勾配リセット**。

## 2-3. `torch.no_grad()` と `detach()`：追跡を止める

- 学習率を掛けてパラメータを更新する操作まで微分対象にすると困る＆無駄なので、`with torch.no_grad():` で囲みます。
- 推論（予測）時も勾配は不要なので `no_grad` で囲むと**速く・省メモリ**になります。
- 計算グラフから値だけ取り出したいときは `.detach()`。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2

with torch.no_grad():
    z = x * 5            # この計算は追跡されない
print("z.requires_grad =", z.requires_grad)   # False

val = y.detach()         # y から勾配の繋がりを切った値
print("detach:", val, val.requires_grad)

## 2-4. なぜ毎回 `zero_()`（勾配リセット）が要るのか

PyTorch は `.backward()` のたびに勾配を **足し込み（累積）** ます。リセットしないと前回の勾配が残って学習が壊れます。
（後の章では `optimizer.zero_grad()` がこの役割を担います。）

In [ ]:
w = torch.tensor(1.0, requires_grad=True)

for i in range(3):
    y = w * 2          # dy/dw = 2
    y.backward()
    print(f"リセットなし {i}: w.grad = {w.grad.item()}")  # 2,4,6 と累積していく！

print("--- 正しくは毎回リセット ---")
w.grad.zero_()
for i in range(3):
    y = w * 2
    y.backward()
    print(f"リセットあり {i}: w.grad = {w.grad.item()}")   # 常に 2
    w.grad.zero_()

## 演習 2
1. $f(x)=(x+5)^2$ を勾配降下で最小化し、$x$ が $-5$ に近づくのを確認しよう。
2. 学習率 `lr` を `0.01` や `0.9`、`1.1` に変えるとどうなる？（小さすぎ＝遅い、大きすぎ＝発散）
3. `y = w**3` の勾配を `w=2` で求め、手計算 $3w^2=12$ と一致するか確認しよう。

In [ ]:
# ここに自分のコードを書いて実行してみよう
